# 01 — Bronze Ingestion

Loads the licensed or synthetic customer-churn CSV into a governed Delta Bronze table and records ingestion metrics.

## Publication notes

- This public notebook contains no credentials, tokens, tenant IDs, subscription IDs, or workspace URLs.
- Notebook outputs were removed to avoid publishing source records or environment metadata.
- Configure the catalog, schema, and source data location for your own Azure environment before execution.
- Run notebooks in numerical order.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
import uuid

## Configuration

Replace `<container>` and `<storage-account>` with resources you control. Authentication should use managed identity or another secretless method; do not place keys or tokens in this notebook.


In [ ]:
CATALOG = "dbx_niw_analytics"
SCHEMA = "ztlf"

TABLE_NAME = "bronze_customer_churn"

SOURCE_FILE = "abfss://<container>@<storage-account>.dfs.core.windows.net/Churn_Modelling.csv"

BATCH_ID = str(uuid.uuid4())

print(BATCH_ID)

In [ ]:
source_schema = StructType([
    StructField("RowNumber", IntegerType(), True),
    StructField("CustomerId", LongType(), True),
    StructField("Surname", StringType(), True),
    StructField("CreditScore", IntegerType(), True),
    StructField("Geography", StringType(), True),
    StructField("Gender", StringType(), True),
    StructField("Age", IntegerType(), True),
    StructField("Tenure", IntegerType(), True),
    StructField("Balance", DoubleType(), True),
    StructField("NumOfProducts", IntegerType(), True),
    StructField("HasCrCard", IntegerType(), True),
    StructField("IsActiveMember", IntegerType(), True),
    StructField("EstimatedSalary", DoubleType(), True),
    StructField("Exited", IntegerType(), True)
])

In [ ]:
bronze_df = (
    spark.read
         .format("csv")
         .option("header", "true")
         .schema(source_schema)
         .load(SOURCE_FILE)
)

display(bronze_df)

In [ ]:
expected_columns = [field.name for field in source_schema.fields]
actual_columns = bronze_df.columns

missing_columns = sorted(set(expected_columns) - set(actual_columns))
unexpected_columns = sorted(set(actual_columns) - set(expected_columns))

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

if unexpected_columns:
    raise ValueError(f"Unexpected columns: {unexpected_columns}")

row_count = bronze_df.count()

print("Schema validation passed.")
print(f"Column count: {len(actual_columns)}")
print(f"Row count: {row_count}")

In [ ]:
business_columns = bronze_df.columns

bronze_enriched_df = (
    bronze_df
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_ingestion_date", F.current_date())
    .withColumn("_ingestion_batch_id", F.lit(BATCH_ID))
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_source_system", F.lit("BankingChurnCSV"))
    .withColumn("_processing_layer", F.lit("BRONZE"))
    .withColumn("_record_status", F.lit("RAW"))
    .withColumn(
        "_record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                *[
                    F.coalesce(
                        F.col(column).cast("string"),
                        F.lit("<NULL>")
                    )
                    for column in business_columns
                ]
            ),
            256
        )
    )
)

display(bronze_enriched_df.limit(10))

In [ ]:
duplicate_count = (
    bronze_enriched_df
    .groupBy(*business_columns)
    .count()
    .filter(F.col("count") > 1)
    .select(
        F.coalesce(
            F.sum(F.col("count") - 1),
            F.lit(0)
        ).alias("duplicate_count")
    )
    .first()["duplicate_count"]
)

null_expressions = [
    F.sum(F.col(column).isNull().cast("int")).alias(column)
    for column in business_columns
]

null_counts = (
    bronze_enriched_df
    .agg(*null_expressions)
    .first()
    .asDict()
)

total_null_values = sum(null_counts.values())

distinct_customer_ids = (
    bronze_enriched_df
    .select("CustomerId")
    .distinct()
    .count()
)

print(f"Total rows: {row_count}")
print(f"Duplicate business rows: {duplicate_count}")
print(f"Total null values: {total_null_values}")
print(f"Distinct customer IDs: {distinct_customer_ids}")

In [ ]:
FULL_TABLE_NAME = f"{CATALOG}.{SCHEMA}.{TABLE_NAME}"

(
    bronze_enriched_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(FULL_TABLE_NAME)
)

print(f"Bronze table created: {FULL_TABLE_NAME}")

In [ ]:
metrics_data = [
    ("source_row_count", int(row_count)),
    ("source_column_count", int(len(business_columns))),
    ("duplicate_business_rows", int(duplicate_count)),
    ("total_null_values", int(total_null_values)),
    ("distinct_customer_ids", int(distinct_customer_ids))
]

metrics_df = (
    spark.createDataFrame(
        metrics_data,
        ["metric_name", "metric_value"]
    )
    .withColumn("experiment_stage", F.lit("BRONZE_INGESTION"))
    .withColumn("dataset_name", F.lit("Churn_Modelling.csv"))
    .withColumn("ingestion_batch_id", F.lit(BATCH_ID))
    .withColumn("measured_at", F.current_timestamp())
)

display(metrics_df)

METRICS_TABLE = f"{CATALOG}.{SCHEMA}.experiment_metrics"

(
    metrics_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(METRICS_TABLE)
)

print(f"Metrics written to: {METRICS_TABLE}")

In [ ]:
spark.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT CustomerId) AS distinct_customer_ids,
    COUNT(DISTINCT _ingestion_batch_id) AS ingestion_batches,
    MIN(_ingestion_timestamp) AS first_ingestion_time,
    MAX(_ingestion_timestamp) AS last_ingestion_time
FROM {FULL_TABLE_NAME}
""").show(truncate=False)

spark.sql(f"DESCRIBE TABLE {FULL_TABLE_NAME}").show(truncate=False)

In [ ]:
spark.sql(f"DESCRIBE HISTORY {FULL_TABLE_NAME}").show(truncate=False)